# Basic comparsion of naive and fast convolution

In [1]:
import numpy as np
import sympy as sy
from scipy import signal
from scipy import datasets
from matplotlib import pyplot as plt

In [2]:
from naive_convolve import naive_convolve
from fast_convolution import C3x3_5m20a9e
from utils import plot_pdf, symmetrical_cyclic_convolution

Lets do a really basic test to compare naive with fast convolution
The input feature will be a simple 5x5 matrix and the output will be a 3x3 matrix

In [3]:
feature = np.arange(5*5).reshape(5, 5)
feature

array([[ 0,  1,  2,  3,  4],
       [ 5,  6,  7,  8,  9],
       [10, 11, 12, 13, 14],
       [15, 16, 17, 18, 19],
       [20, 21, 22, 23, 24]])

In [4]:
weight = np.array([
    [ 1, 0, 1],
    [ 2, 1, 2],
    [ 1, 2, 1],
])
weight

array([[1, 0, 1],
       [2, 1, 2],
       [1, 2, 1]])

In [5]:
np.fliplr(weight)

array([[1, 0, 1],
       [2, 1, 2],
       [1, 2, 1]])

Convolution with no paddin and stride 1
Using convolve2d from scipy, our gold method, it's necessary to reverse the weights to get same result of naive and fast convolution

In [34]:
weight_reversed = weight[::-1, ::-1]
weight_reversed

array([[1, 2, 1],
       [2, 1, 2],
       [1, 0, 1]])

In [35]:
output = signal.convolve2d(feature, weight_reversed, mode='valid')
output

array([[ 76,  87,  98],
       [131, 142, 153],
       [186, 197, 208]])

Running naive convolution
9 multiplications and 8 aditions per output scalar

In [7]:
output_naive = naive_convolve(feature, weight)
output_naive

array([[ 76,  87,  98],
       [131, 142, 153],
       [186, 197, 208]])

In [39]:
np.all(output == output_naive)

True

Fast convolution need to reverse the features order

In [8]:
fv = list(reversed(feature[0].tolist()))
fv

[4, 3, 2, 1, 0]

Init Tap filter from fast 1d convolution method with 5 multiplications, 20 aditions and 9 extras operations, 5 input and 3 output per batch

In [10]:
fast_conv = C3x3_5m20a9e(weight[0].tolist())

Tap filter work in batch mode and the output is reversed

In [45]:
fast_conv(fv[2:] + [0, 0])

Matrix([
[2],
[1],
[0]])

In [49]:
fast_conv([0] + fv[:-1])

Matrix([
[3],
[6],
[4]])

How join multiple 1d convolution in one 2d convolution

In [12]:
c0 = np.convolve(feature[0], weight[0])
c0

array([0, 1, 2, 4, 6, 3, 4])

In [13]:
c1 = np.convolve(feature[1], weight[1])
c1

array([10, 17, 30, 35, 40, 25, 18])

In [14]:
c2 = np.convolve(feature[2], weight[2])
c2

array([10, 31, 44, 48, 52, 41, 14])

In [15]:
c0+c1+c2

array([20, 49, 76, 87, 98, 69, 36])

Run one fast convolution for each 1d kernel

In [16]:
fast_conv0 = C3x3_5m20a9e(weight[0].tolist())
out0 = [fast_conv0(feature[i]).flat() for i in range(0, 3)]
out0

[[2, 4, 6], [12, 14, 16], [22, 24, 26]]

In [17]:
fast_conv1 = C3x3_5m20a9e(weight[1].tolist())
out1 = [fast_conv1(feature[i+1]).flat() for i in range(0, 3)]
out1

[[30, 35, 40], [55, 60, 65], [80, 85, 90]]

In [18]:
fast_conv2 = C3x3_5m20a9e(weight[2].tolist())
out2 = [fast_conv2(feature[i+2]).flat() for i in range(0, 3)]
out2

[[44, 48, 52], [64, 68, 72], [84, 88, 92]]

Sum results in the first dimension

In [19]:
output_fast = np.sum([out0, out1, out2], axis=0)
output_fast

array([[76, 87, 98],
       [131, 142, 153],
       [186, 197, 208]], dtype=object)

In [40]:
np.all(output_fast == output_naive)

True

Camparing how much operations are used in naive and fast method

Output Size

In [21]:
size = output.size
size

9

Naive: total of multiplications

In [22]:
size * 9

81

Naive: total of additions

In [23]:
size * 8

72

Fast: total of multiplications

In [24]:
size * 5

45

Fast: additions for each batch processed

In [25]:
add0 = size * 20
add0

180

Fast: additions to join batches

In [26]:
add1 = size * 2
add1

18

Fast: Total of additions

In [27]:
add0 + add1

198

Fast: total of extra operations - bit shifts and etc

In [50]:
size * 9

81